# Задачи по очистке данных

Версия для слушателей.

## Что проверяется

- диагностика качества данных;
- обработка пропусков;
- преобразование дат;
- очистка текстовых значений;
- удаление дубликатов;
- проверка бизнес-правил;
- формирование карантина;
- обнаружение выбросов методом IQR;
- обнаружение выбросов методом z-score;
- объединение таблиц через `merge()`;
- валидация данных через `assert`;
- формирование отчёта качества.

## Правила работы

1. Выполните служебную ячейку подготовки окружения.
2. Каждая задача самодостаточна: в ней заново создаётся нужная таблица.
3. Под формулировкой задания находится отдельная ячейка для решения.
4. Не удаляйте исходные данные.
5. После выполнения запустите notebook через **Run all**.
6. Основные задания — 1–16. Задания 17–20 можно использовать как повышенный уровень.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

def make_sales_data():
    """Возвращает небольшую таблицу продаж с контролируемыми ошибками качества."""
    return pd.DataFrame(
        {
            "order_id": [
                "ORD-001", "ORD-002", "ORD-003", "ORD-004", "ORD-005",
                "ORD-006", "ORD-007", "ORD-008", "ORD-009", "ORD-010",
                "ORD-011", "ORD-012", "ORD-013", "ORD-014", "ORD-015",
                "ORD-015",
            ],
            "order_date": [
                "2026-04-01", "2026-04-02", "not_a_date", "2026-04-04",
                "2026-04-05", "2026-04-06", "2026/99/99", "2026-04-08",
                "2026-04-09", "2026-04-10", "2026-04-11", "2026-04-12",
                "2026-04-13", "2026-04-14", "2026-04-15", "2026-04-15",
            ],
            "client_id": [
                "C001", "C002", "C003", "C004", " C005 ",
                "C006", "C007", "C008", "C009", "C010",
                "C011", "C012", "C013", "C014", "C015", "C015",
            ],
            "product_id": [
                "P001", "P002", "P003", "P004", "P005",
                "P001", "P002", "P999", " P003 ", "P004",
                "P005", "P001", "P002", "P003", "P004", "P004",
            ],
            "region_id": [
                "R01", "R02", np.nan, "R03", "R01",
                "R02", "R03", "R99", " R01 ", "R02",
                "R03", "R01", "R02", "R03", np.nan, np.nan,
            ],
            "channel": [
                "online", "Online", " ONLINE ", "marketplace", " Marketplace ",
                "offline", "Offline ", "partner", "PARTNER ", "online",
                "marketplace", "offline", "ONLINE", "partner", "online", "online",
            ],
            "quantity": [
                2, 5, 3, -1, 4,
                0, 6, 2, 3, 1,
                25, 2, 4, 3, 2, 2,
            ],
            "unit_price": [
                70000, 1500, 3500, 72000, 3400,
                1450, 0, 2500, 3600, -100,
                3800, 71000, 1550, 150000, 74000, 74000,
            ],
            "discount": [
                0.10, np.nan, 0.05, 0.00, 0.10,
                0.15, 0.20, 0.05, 1.30, -0.10,
                0.00, 0.10, np.nan, 0.05, 0.10, 0.10,
            ],
            "delivery_days": [
                3, 5, np.nan, 4, 6,
                2, 8, np.nan, 5, 4,
                7, 3, 6, 2, np.nan, np.nan,
            ],
        }
    )

def make_products():
    """Справочник товаров с уникальным ключом product_id."""
    return pd.DataFrame(
        {
            "product_id": ["P001", "P002", "P003", "P004", "P005"],
            "product_name": [
                "Ноутбук", "Мышь", "Клавиатура", "Монитор", "Наушники"
            ],
            "category": [
                "Компьютеры", "Аксессуары", "Аксессуары", "Мониторы", "Аудио"
            ],
            "cost": [52000, 800, 2100, 48000, 2300],
        }
    )

def make_regions():
    """Справочник регионов."""
    return pd.DataFrame(
        {
            "region_id": ["R01", "R02", "R03"],
            "region_name": ["Москва", "Казань", "Самара"],
        }
    )

print("Окружение подготовлено")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

Окружение подготовлено
pandas: 2.2.3
numpy: 2.3.5


# Раздел 1. Первичная диагностика

## Задача 1. Структура таблицы


Создайте таблицу `sales` через `make_sales_data()`.

Выведите:

1. первые пять строк;
2. размер таблицы;
3. список столбцов;
4. типы данных.

**Ожидаемый результат:** несколько объектов вывода, позволяющих описать структуру таблицы.

In [2]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 2. Пропуски по столбцам


Посчитайте количество пропущенных значений в каждом столбце.

Дополнительно рассчитайте долю пропусков в процентах.

**Подсказка:** используйте `isna().sum()` и деление на количество строк.

In [3]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 3. Поиск дубликатов


Найдите:

1. количество полных дубликатов строк;
2. количество повторных `order_id`;
3. сами строки с повторяющимся `order_id`.

Используйте `duplicated()`.

In [4]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 4. Поиск несогласованных текстовых значений


Исследуйте столбец `channel`.

Выведите:

- уникальные значения;
- частоту каждого значения.

Определите, какие проблемы качества видны в этом столбце.

In [5]:
sales = make_sales_data()

# Напишите решение ниже

# Раздел 2. Обработка пропусков

## Задача 5. Заполнение категориального пропуска


Заполните пропуски в `region_id` значением `UNKNOWN`.

Проверьте количество пропусков до и после обработки.

In [6]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 6. Заполнение числового пропуска средним


Заполните пропуски в `delivery_days` средним значением столбца.

Перед заполнением выведите среднее и медиану. После заполнения убедитесь, что пропусков нет.

In [7]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 7. Заполнение пропуска в скидке


Заполните пропуски в `discount` значением `0`.

Объясните в комментарии, какой бизнес-смысл имеет такое решение.

In [8]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 8. Некорректные даты


Преобразуйте `order_date` в формат даты через `pd.to_datetime()` с `errors="coerce"`.

Затем:

1. найдите строки, где дата стала `NaT`;
2. удалите эти строки;
3. выведите количество строк до и после удаления.

In [9]:
sales = make_sales_data()

# Напишите решение ниже

# Раздел 3. Очистка текста, дубликатов и бизнес-ошибок

## Задача 9. Нормализация текста


Приведите `channel` к единому виду:

- удалите пробелы по краям;
- переведите значения в нижний регистр.

Сравните уникальные значения до и после.

In [10]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 10. Очистка ключей


Удалите лишние пробелы в столбцах:

- `client_id`;
- `product_id`;
- `region_id`.

Учтите, что в `region_id` есть пропуски.

In [11]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 11. Удаление дубликатов


Удалите повторный заказ по `order_id`, сохранив первую запись.

Выведите количество строк до и после и проверьте уникальность `order_id`.

In [12]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 12. Проверка бизнес-правил


Найдите строки, нарушающие правила:

- `quantity > 0`;
- `unit_price > 0`;
- `0 <= discount <= 1`.

Сформируйте три отдельные таблицы проблемных строк.

In [13]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 13. Карантин проблемных строк


Создайте таблицу `quarantine`.

В неё должны попасть строки с:

- неположительным количеством;
- неположительной ценой;
- скидкой вне диапазона.

Добавьте столбец `error_reason`, в котором указана причина.

In [14]:
sales = make_sales_data()

# Напишите решение ниже

# Раздел 4. Валидация и отчёт качества

## Задача 14. Очистка по бизнес-правилам


Создайте `sales_clean`, оставив только строки, где:

- количество положительное;
- цена положительная;
- скидка находится от 0 до 1 или является пропуском.

Затем заполните пропуски скидки нулём.

In [15]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 15. Валидация через `assert`


Подготовьте очищенную таблицу:

1. преобразуйте даты;
2. удалите строки с плохими датами;
3. удалите дубликаты `order_id`;
4. оставьте только допустимые количество, цену и скидку;
5. заполните пропуски скидки нулём.

После этого добавьте проверки `assert` для всех правил.

In [16]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 16. Отчёт качества до и после


Сформируйте таблицу `quality_report` с показателями:

- количество строк;
- число пропусков;
- число повторных `order_id`;
- число некорректных дат;
- число строк с `quantity <= 0`;
- число строк с `unit_price <= 0`;
- число скидок вне диапазона.

Для каждого показателя добавьте значения `before` и `after`.

In [17]:
sales = make_sales_data()

# Подготовьте sales_clean и сформируйте quality_report
# Напишите решение ниже

# Раздел 5. Выбросы

## Задача 17. Поиск выбросов методом IQR

Подготовьте положительные значения `quantity`.

Рассчитайте:

- Q1;
- Q3;
- IQR;
- нижнюю и верхнюю границы;
- таблицу потенциальных выбросов.

Не удаляйте найденные строки.


In [18]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 18. Поиск выбросов методом z-score

Для положительных значений `quantity` рассчитайте z-score.

Добавьте столбец `quantity_z_score` и выберите строки, где абсолютное значение z-score больше 3.


In [19]:
sales = make_sales_data()

# Напишите решение ниже

## Задача 19. Сравнение IQR и z-score

Создайте два логических столбца:

- `is_outlier_iqr`;
- `is_outlier_zscore`.

Затем выберите все строки, которые отметил хотя бы один метод.

Сформулируйте в комментарии, почему такие строки нельзя удалять автоматически.


In [20]:
sales = make_sales_data()
sales = sales[sales["quantity"] > 0].copy()

# Напишите решение ниже

# Раздел 6. Объединение и итоговый мини-кейс

## Задача 20. `merge()` и проверка результата


Очистите пробелы в `product_id` и объедините продажи со справочником товаров.

Требования:

1. используйте `how="left"`;
2. добавьте `validate="many_to_one"`;
3. сравните число строк до и после;
4. найдите заказы, для которых товар не найден.

In [21]:
sales = make_sales_data()
products = make_products()

# Напишите решение ниже

# Итоговый чек-лист

- [ ] Все основные задачи выполнены.
- [ ] Notebook запускается через **Run all**.
- [ ] Пропуски обработаны обоснованным способом.
- [ ] Дубликаты удалены по явно указанному ключу.
- [ ] Бизнес-ошибки отделены от статистических выбросов.
- [ ] Выбросы не удаляются автоматически.
- [ ] После `merge()` проверено количество строк и ненайденные ключи.
- [ ] Валидация через `assert` проходит без ошибок.